In [ ]:
import xarray as xr
import numpy as np
from scipy.fft import fft, ifft, fftfreq, fft2, ifft2, fftshift
import pandas as pd
from scipy.signal import detrend
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

In [ ]:
#code based on WK99

def pre_process(data, lat, var = None ):
    #detrend
    uw = data[var].sel(latitude = slice(lat, -lat))
    # uw = uw.drop_vars({"run_number", "init_time", "valid_time"})
    mean_uw = uw.mean(dim="time")
    trend = uw.polyfit(dim="time", deg=1)
    fitted = xr.polyval(uw["time"], trend.polyfit_coefficients)
    detrended = uw - fitted
    # detrended += mean_uw
    
  
    #taper
    hann = split_hann_taper(detrended.shape[0], 0.05)
    data_tapered = detrended * hann[:, None, None]
    hann_lon = split_hann_taper(data_tapered.shape[2],0.05)
    data_tapered *= hann_lon[None, None, :]
    
    return data_tapered

def split_hann_taper(series_length, fraction=0.05):
   
    npts = int(np.rint(series_length * fraction))
    window = np.hanning(npts)
    half = npts // 2
    
    taper = np.ones(series_length)
    taper[:half] = window[:half]         
    taper[-half:] = window[-half:]       
    # taper = taper / np.sqrt(np.mean(taper**2))
    return taper
    
def sym_asym(data):

    lat_dim = data.get_axis_num("latitude")
    flipped = np.flip(data.values, axis=lat_dim)
    sym  = 0.5 * (data.values + flipped)
    asym = 0.5 * (data.values - flipped)

    return sym, asym

def wk_fft(field, dt, dlon):
 
    nt,_, nlon = field.shape
    F = fft2(field, axes=(0,2))
    power = np.real(F * np.conj(F))
    power = fftshift(power, axes=(0,2))
    power = np.flip(power,axis=0)

    kt = fftshift(fftfreq(nt, dt) )       
    kx = fftshift(fftfreq(nlon, dlon/nlon))   
    power = np.nanmean(power, axis=1)
    izero = np.argmin(np.abs(kt))
    power[izero, :] = power[izero+1, :]
    
    return power, kx, kt


# logic borrowed from https://github.com/suhasdl/pySWE/blob/master/plotUtils/wheelerKiladis.py
def filter121(data):
    weights = np.array((0.25,0.5,0.25))
    mavg = np.convolve(data,weights,'same')
    return mavg  

def apply_filter(dsym,dasym,kx,kt):

    for i in range(5):
        dsym[:,(abs(kx)<=27.)]  = np.apply_along_axis(filter121,0,dsym[:,(abs(kx)<=27.)])
        dasym[:,(abs(kx)<=27.)] = np.apply_along_axis(filter121,0,dasym[:,(abs(kx)<=27.)])
    dtot = (dasym+ dsym)/2

    for j in range(5):
        idx = (abs(kt)<= 0.1)
        dtot[idx,:]  = np.apply_along_axis(filter121,1,dtot[idx,:])
    for j in range(10):
        idx = np.logical_and(abs(kt)>0.1,abs(kt)<=0.2)
        dtot[idx,:]  = np.apply_along_axis(filter121,1,dtot[idx,:])
    for j in range(12):
        idx = np.logical_and(abs(kt)>0.2,abs(kt)<=0.3)
        dtot[idx,:]  = np.apply_along_axis(filter121,1,dtot[idx,:])
    for j in range(15):
        idx = (abs(kt) > 0.3)
        dtot[idx,:]  = np.apply_along_axis(filter121,1,dtot[idx,:])
    for j in range(5):
        jdx = (abs(kx)<=27.)
        dtot[:,jdx]  = np.apply_along_axis(filter121,0,dtot[:,jdx])
    return dtot,dsym,dasym


    
def equatorial_wave_curves():

    g = 9.81
    beta = 2.28e-11      # equatorial beta (m^-1 s^-1)
    re = 6.371e6         # Earth radius (m)

    k = np.arange(-15, 16)# zonal wavenumbers
    he = [12, 25, 50,]# equivalent depths (m)
    curves = {}
    for h in he:
        c = np.sqrt(g * h)
        ld = np.sqrt(c / beta)
        kn = k / re 
        omega_kelvin = c * kn
        omega_er = -beta * kn / (kn**2 + (3 / ld**2))# ER waves (n=1)
        omega_mrg = 0.5 * c * kn+ 0.5 * np.sqrt((c * kn)**2 + 4 * beta * c)# MRG wave
        omega_ig1 = np.sqrt((c * kn)**2 + 3 * beta * c)# IG waves (n=1)
        omega_ig2 = np.sqrt((c * kn)**2 + 5 * beta * c)# IG waves (n=2)
        cpd = 86400 / (2 * np.pi)# convert to cycles/day
        curves[h] = {
            'k': k,
            'kelvin': omega_kelvin * cpd,
            'er': np.abs(omega_er) * cpd,
            'mrg': np.abs(omega_mrg) * cpd,
            'ig1': omega_ig1 * cpd,
            'ig2': omega_ig2 * cpd
        }
    return curves


In [ ]:
all_sym = []
all_asym = []
all_ptot = []
for f in np.arange(1980, 1980+34, 1):

    ds = xr.open_dataset(f"/data/data/era5/toa_lwr_1deg/toa_lwr_{f}.nc")
    ds = ds.isel(time = slice(0, 120*4))

    dt = 0.25
    dlon = 1

    data_tapered = pre_process(ds, 15, var = 'avg_tnlwrf')

    sym, asym = sym_asym(data_tapered)

    Psym, kx, kt = wk_fft(sym, dt, dlon)
    Pasym, _, _ = wk_fft(asym, dt, dlon)
    Ptot, _, _ = wk_fft(data_tapered.values, dt, dlon)
    ptot, psym, pasym  = apply_filter(Psym, Pasym, kx, kt)
    all_sym.append(psym)
    all_asym.append(pasym)
    all_ptot.append(ptot)
    
psym = np.nanmean(all_sym, axis=0)
pasym = np.nanmean(all_asym, axis=0)
ptot = np.nanmean(all_ptot, axis=0)

power_sym = np.log10(psym)
power_asym = np.log10(pasym)
power_ptot = np.log10(ptot)

curves = equatorial_wave_curves()

In [ ]:
all_sym = []
all_asym = []
all_ptot = []

for f in range(34):

    ds = xr.open_dataset(f"/media/shrutee/Expansion/ai_models/new_lr/fc2_{f}.nc").sel(run_number = f, level = 850)
    
    dt = 0.25
    dlon = 1.0

    data_tapered = pre_process(ds, 15, var = 'u_wind')

    sym, asym = sym_asym(data_tapered)

    Psym, kx, kt = wk_fft(sym, dt, dlon)
    Pasym, _, _ = wk_fft(asym, dt, dlon)
    Ptot, _, _ = wk_fft(data_tapered, dt, dlon)
    ptot, psym, pasym  = apply_filter(Psym, Pasym, kx, kt)
    all_sym.append(psym)
    all_asym.append(pasym)
    all_ptot.append(ptot)

psym = np.nanmean(all_sym, axis=0)
pasym = np.nanmean(all_asym, axis=0)
ptot = np.nanmean(all_ptot, axis=0)

power_sym = np.log10(psym)
power_asym = np.log10(pasym)
power_ptot = np.log10(ptot)

curves = equatorial_wave_curves()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8, 4), sharey=True, constrained_layout = True)

levels = np.arange(0.99, 1.045, 0.002)


cf1 = ax[0].contourf(
    kx, kt, power_sym / power_ptot,
     levels = levels,
    extend='both',
    cmap='Spectral_r'
)

# ax[0].set_title("Symmetric")
ax[0].set_xlim(-15, 15)
ax[0].set_ylim(0.0, 0.8)

for h, cd in curves.items():
    k = cd['k']
    ax[0].plot(k, cd['kelvin'], 'k--', lw=1)
    ax[0].plot(k[k <= 0], cd['er'][k <= 0], 'k--', lw=1)
    ax[0].plot(k, cd['ig1'], 'k--', lw=1)


cf2 = ax[1].contourf(
    kx, kt, power_asym / power_ptot,
    levels = levels,
    extend='both',
    cmap='Spectral_r'
)

# ax[1].set_title("Asymmetric")
ax[1].set_xlim(-15, 15)
ax[1].set_ylim(0.0, 0.8)

for h, cd in curves.items():
    k = cd['k']
    ax[1].plot(k, cd['ig2'], 'k--', lw=1)
    ax[1].plot(k, cd['mrg'], 'k--', lw=1)


fig.subplots_adjust(right=0.88)

cbar_ax = fig.add_axes([1, 0.15, 0.02, 0.7])
cbar = fig.colorbar(cf2, cax=cbar_ax)
# cbar.set_ticks(np.arange(1, 1.04,0.01))
# plt.tight_layout()
# plt.savefig("/media/shrutee/Expansion/ai_models/new_lr/filtered/wk_plots/era5_olr.png", dpi = 300, bbox_inches = 'tight')
# plt.show()

In [ ]:
c = plt.contourf(kx,kt,power_ptot, levels = np.arange(8, 14,0.5),  extend = 'both',  cmap = 'Reds')
plt.colorbar(c)
plt.xlim(-10, 10)
plt.ylim(0.0, 0.8)

In [ ]:
c = plt.contourf(kx,kt,power_sym,  levels = np.arange(7,11,0.5),  extend = 'both',  cmap = 'Blues')
plt.colorbar(c)
plt.xlim(-10, 10)
plt.ylim(0.0, 0.8)

In [ ]:
c = plt.contourf(kx,kt,power_asym, levels = np.arange(7,11,0.5),  extend = 'both',  cmap = 'Blues')
plt.colorbar(c)
plt.xlim(-10, 10)
plt.ylim(0.0, 0.8)